# 01 — EDA: Medical Cost Personal Dataset

**Bài toán:** hồi quy `charges` (USD). Nguồn Kaggle `mirichoi0218/insurance`.

Mỗi hình dưới đây trả lời 3 câu: hình cho thấy gì / ý nghĩa / quyết định xử lý (nối sang notebook 02).

Chạy **Restart & Run All**. Đường dẫn zip: `../data/dataset.zip`.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
ROOT = Path("..")
ZIP = ROOT / "data" / "dataset.zip"
assert ZIP.exists(), f"Không thấy {ZIP.resolve()} — hãy chạy notebook từ ai-models/colab/"

with ZipFile(ZIP) as zf:
    name = next(n for n in zf.namelist() if n.endswith(".csv"))
    df = pd.read_csv(zf.open(name))

print(df.shape)
print(df.dtypes)
print("missing:\n", df.isnull().sum())
print("duplicates:", int(df.duplicated().sum()))
df.head()


## Kiểm tra ban đầu

- 1338 dòng, 7 cột, 0 missing.
- 1 duplicate — quyết định **giữ** để khớp số dòng công bố trên Kaggle.
- Không có cột nào sinh ra từ `charges` → không rò rỉ nhãn.


In [ ]:
df.describe(include="all").T


## Hình 1 — Phân phối `charges`

1. **Cho thấy gì?** Lệch phải, đuôi dài tới ~64k USD, mean > median.
2. **Ý nghĩa?** Hồi quy, sai số tuyệt đối (MAE) dễ giải thích; RMSE nhạy với ca đắt.
3. **Quyết định?** Giữ nguyên đơn vị USD; dùng MAE + RMSE + R².


In [ ]:
plt.figure(figsize=(8,4.5))
sns.histplot(df["charges"], kde=True, bins=40)
plt.title("Hình 1 — Phân phối charges")
plt.show()


## Hình 2 — `smoker` vs `charges`

1. **Cho thấy gì?** Median nhóm hút thuốc cao gấp ~3–4 lần nhóm không hút.
2. **Ý nghĩa?** `smoker` là đặc trưng mạnh nhất.
3. **Quyết định?** One-hot `smoker`, không cắt outlier nhóm yes.


In [ ]:
plt.figure(figsize=(6,4.5))
sns.boxplot(data=df, x="smoker", y="charges")
plt.title("Hình 2 — smoker vs charges")
plt.show()


## Hình 3 — `age` vs `charges`

1. **Cho thấy gì?** Ba dải song song theo tuổi; dải trên là smoker=yes.
2. **Ý nghĩa?** Có tương tác age × smoker.
3. **Quyết định?** Baseline tuyến tính + model cây/ensemble.


In [ ]:
plt.figure(figsize=(6.5,4.5))
sns.scatterplot(data=df, x="age", y="charges", hue="smoker", alpha=0.7)
plt.title("Hình 3 — age vs charges")
plt.show()


## Hình 4 — `bmi` vs `charges`

1. **Cho thấy gì?** BMI cao + hút thuốc → charges nhảy vọt.
2. **Ý nghĩa?** Tương tác phi tuyến bmi × smoker.
3. **Quyết định?** Không cắt BMI; để cây tách nhánh.


In [ ]:
plt.figure(figsize=(6.5,4.5))
sns.scatterplot(data=df, x="bmi", y="charges", hue="smoker", alpha=0.7)
plt.title("Hình 4 — bmi vs charges")
plt.show()


## Hình 5 — `region` vs `charges`

1. **Cho thấy gì?** Median 4 vùng gần nhau.
2. **Ý nghĩa?** region yếu hơn smoker/age/bmi.
3. **Quyết định?** One-hot, không gộp vùng.


In [ ]:
plt.figure(figsize=(7,4.5))
sns.boxplot(data=df, x="region", y="charges")
plt.title("Hình 5 — region vs charges")
plt.show()


## Hình 6 — Missing heatmap · Hình 7 — Tương quan


In [ ]:
fig, ax = plt.subplots(figsize=(8,3))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, ax=ax)
ax.set_title("Hình 6 — Bản đồ missing (toàn 0)")
plt.show()

tmp = df.copy()
tmp["smoker_yes"] = (tmp.smoker=="yes").astype(int)
tmp["sex_male"] = (tmp.sex=="male").astype(int)
plt.figure(figsize=(7,5.5))
sns.heatmap(tmp[["age","bmi","children","smoker_yes","sex_male","charges"]].corr(),
            annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Hình 7 — Heatmap tương quan")
plt.show()
